In [3]:
# Fase 6: Chunking y Fragmentación (Gold Layer)

StatementMeta(, 815d997b-424f-4fb5-b342-4441af025c61, 5, Finished, Available, Finished, False)

In [4]:
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, TimestampType
)

# Esquema oficial para gold_document_chunks
schema_gold_chunks = StructType([
    StructField("chunk_id", StringType(), False),
    StructField("document_id", StringType(), False),
    StructField("chunk_number", IntegerType(), False),
    StructField("chunk_text", StringType(), False),
    StructField("chunk_size", IntegerType(), False),       # Número de caracteres/tokens del chunk
    StructField("document_category", StringType(), False),
    StructField("document_title", StringType(), True),
    StructField("source_url", StringType(), True),
    StructField("page_number", IntegerType(), True),      # Estimación de página asociada
    StructField("created_at", TimestampType(), False)
])

# Crear la tabla Delta vacía en la Capa Gold (si no existe)
spark.createDataFrame([], schema_gold_chunks).write.format("delta").mode("ignore").saveAsTable("gold_document_chunks")

print("Tabla 'gold_document_chunks' verificada e inicializada en LH_Ecodocs.")

StatementMeta(, 815d997b-424f-4fb5-b342-4441af025c61, 6, Finished, Available, Finished, False)

Tabla 'gold_document_chunks' verificada e inicializada en LH_Ecodocs.


In [6]:
import os
import hashlib
from datetime import datetime
from pyspark.sql import Row
from pyspark.sql.functions import col

# Intentar usar el splitter recursivo de LangChain si está disponible
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    USE_LANGCHAIN = True
except ImportError:
    USE_LANGCHAIN = False

# ---------------------------------------------------------
# FUNCIONES DE CHUNKING CON OVERLAP
# ---------------------------------------------------------

def chunk_text_native(text: str, chunk_size_chars: int = 2500, overlap_chars: int = 500):
    """
    Función de fallback nativa en Python si no está instalado LangChain.
    Divide el texto respetando un número de caracteres con solapamiento (overlap).
    """
    chunks = []
    start = 0
    text_len = len(text)
    
    while start < text_len:
        end = start + chunk_size_chars
        chunk = text[start:end]
        
        # Ajustar para no cortar palabras a la mitad si no estamos al final
        if end < text_len:
            last_space = chunk.rfind(' ')
            if last_space != -1 and last_space > (chunk_size_chars * 0.7):
                end = start + last_space
                chunk = text[start:end]
                
        chunks.append(chunk.strip())
        start = end - overlap_chars
        
        # Evitar bucles infinitos si el avance es menor que el overlap
        if start >= end:
            start = end
            
    return [c for c in chunks if c]

def split_document_into_chunks(text: str, target_tokens: int = 700, overlap_tokens: int = 120):
    """
    Convierte tokens aproximados a caracteres (1 token ≈ 4 caracteres en español).
    Retorna una lista de strings con los chunks generados.
    """
    if not text:
        return []
        
    char_size = target_tokens * 4   # ~2800 caracteres (~700 tokens)
    char_overlap = overlap_tokens * 4 # ~480 caracteres (~120 tokens)
    
    if USE_LANGCHAIN:
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=char_size,
            chunk_overlap=char_overlap,
            separators=["\n\n", "\n", ". ", " ", ""]
        )
        return splitter.split_text(text)
    else:
        return chunk_text_native(text, chunk_size_chars=char_size, overlap_chars=char_overlap)

# ---------------------------------------------------------
# PROCESAMIENTO Y GENERACIÓN DE CHUNKS EN GOLD
# ---------------------------------------------------------


# 1. Leer solo documentos procesables (valid o warning) de silver_documents
valid_documents_df = spark.table("silver_documents") \
    .filter(col("quality_status").isin(["valid", "warning"])) \
    .collect()

print(f"Documentos válidos a procesar desde Silver: {len(valid_documents_df)}")

gold_chunk_rows = []

for doc in valid_documents_df:
    doc_id = doc.document_id
    title = doc.document_title
    category = doc.document_category
    source_url = doc.source_url
    clean_text = doc.clean_text
    total_pages = doc.num_pages if hasattr(doc, "num_pages") and doc.num_pages else 1
    
    # Tarea 1: Aplicar la función de chunking
    raw_chunks = split_document_into_chunks(clean_text, target_tokens=700, overlap_tokens=120)
    total_chunks = len(raw_chunks)
    
    for idx, chunk_str in enumerate(raw_chunks, start=1):
        # Generar ID único para el chunk: CHK_MD5HASH_NUM
        chunk_hash = hashlib.md5(f"{doc_id}_{idx}".encode('utf-8')).hexdigest()[:8]
        chunk_id = f"CHK_{chunk_hash}"
        
        # Estimar página asociada según la posición relativa del chunk
        estimated_page = max(1, round((idx / total_chunks) * total_pages)) if total_pages > 1 else 1
        
        gold_chunk_rows.append(Row(
            chunk_id=chunk_id,
            document_id=doc_id,
            chunk_number=idx,
            chunk_text=chunk_str,
            chunk_size=len(chunk_str), # Tamaño en caracteres
            document_category=category,
            document_title=title,
            source_url=source_url,
            page_number=estimated_page,
            created_at=datetime.now()
        ))

# ---------------------------------------------------------
# PERSISTENCIA EN GOLD (OVERWRITE / APPEND IDEMPOTENTE)
# ---------------------------------------------------------

if gold_chunk_rows:
    df_gold_chunks = spark.createDataFrame(gold_chunk_rows, schema=spark.table("gold_document_chunks").schema)
    
    # Sobrescribir la capa Gold para asegurar una sincronización limpia con Silver
    df_gold_chunks.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_document_chunks")
    
    print(f" ¡Éxito! Se han generado y guardado {len(gold_chunk_rows)} chunks en 'gold_document_chunks'.")
else:
    print("No se generaron chunks. Verifica que existan documentos en 'silver_documents' con estado 'valid' o 'warning'.")

StatementMeta(, 815d997b-424f-4fb5-b342-4441af025c61, 8, Finished, Available, Finished, False)

Documentos válidos a procesar desde Silver: 8
 ¡Éxito! Se han generado y guardado 489 chunks en 'gold_document_chunks'.


In [8]:
# Consulta de validación de chunks por documento
spark.sql("""
    SELECT 
        document_title,
        document_category,
        COUNT(chunk_id) AS total_chunks,
        MIN(chunk_size) AS min_chunk_size_chars,
        AVG(chunk_size) AS avg_chunk_size_chars,
        MAX(chunk_size) AS max_chunk_size_chars
    FROM gold_document_chunks
    GROUP BY document_title, document_category
    ORDER BY total_chunks DESC
""").show(truncate=False)

# Muestra de chunks individuales
spark.sql("""
    SELECT chunk_id, document_id, chunk_number, document_category, chunk_size, SUBSTRING(chunk_text, 1, 80) AS preview
    FROM gold_document_chunks
    LIMIT 5
""").show(truncate=False)

StatementMeta(, 815d997b-424f-4fb5-b342-4441af025c61, 10, Finished, Available, Finished, False)

+------------------------------------------------------+-----------------+------------+--------------------+--------------------+--------------------+
|document_title                                        |document_category|total_chunks|min_chunk_size_chars|avg_chunk_size_chars|max_chunk_size_chars|
+------------------------------------------------------+-----------------+------------+--------------------+--------------------+--------------------+
|Esg Ib Informe Sostenibilidad                         |esg              |211         |75                  |2777.9431279620853  |2799                |
|Technical Guia Profesional Tramitacion Autoconsumo V.6|technical        |179         |1174                |2785.7932960893854  |2799                |
|Legal Reglamento 2016-679                             |legal            |94          |815                 |2773.9893617021276  |2799                |
|Hr Politica Teletrabajo Ecopower                      |hr               |1           |429    

In [10]:
from pyspark.sql.functions import col, length, count, avg, min, max

gold_df = spark.table("gold_document_chunks")

print("==================================================")
print("   AUDITORÍA DE CALIDAD Y CONSISTENCIA (GOLD)     ")
print("==================================================")

# 1. Prueba de Registros y Nulos
total_chunks = gold_df.count()
null_chunks = gold_df.filter(col("chunk_text").isNull() | (length(col("chunk_text")) == 0)).count()
null_titles = gold_df.filter(col("document_title").isNull()).count()

print(f" Total de chunks generados: {total_chunks}")
print(f" Chunks vacíos/nulos: {null_chunks} {'(¡OK!)' if null_chunks == 0 else '❌ (Atención)'}")
print(f" Metadatos incompletos (sin título): {null_titles} {'(¡OK!)' if null_titles == 0 else '❌ (Atención)'}")

# 2. Distribución de Chunks por Documento y Categoría
print("\n Resumen de Chunks por Documento:")
spark.sql("""
    SELECT 
        document_category AS categoria,
        document_title AS titulo_documento,
        COUNT(chunk_id) AS total_chunks,
        MIN(chunk_size) AS min_caracteres,
        ROUND(AVG(chunk_size), 0) AS avg_caracteres,
        MAX(chunk_size) AS max_caracteres
    FROM gold_document_chunks
    GROUP BY document_category, document_title
    ORDER BY total_chunks DESC
""").show(truncate=False)

# 3. Comprobación de Solapamiento (Overlap) en un documento
sample_doc = gold_df.select("document_id").first()
if sample_doc:
    doc_id_sample = sample_doc.document_id
    print(f"\n Inspección de continuidad/overlap (Documento ID: {doc_id_sample[:8]}...):")
    
    sample_chunks = gold_df.filter(col("document_id") == doc_id_sample) \
                           .orderBy("chunk_number") \
                           .select("chunk_number", "chunk_size", "chunk_text") \
                           .take(2)
    
    for c in sample_chunks:
        print(f"\n--- CHUNK {c.chunk_number} ({c.chunk_size} caracteres) ---")
        print(f"{c.chunk_text[:120]} ... [TRUNCADO] ... {c.chunk_text[-120:]}")

print("\n==================================================")

StatementMeta(, 815d997b-424f-4fb5-b342-4441af025c61, 12, Finished, Available, Finished, False)

   AUDITORÍA DE CALIDAD Y CONSISTENCIA (GOLD)     
 Total de chunks generados: 489
 Chunks vacíos/nulos: 0 (¡OK!)
 Metadatos incompletos (sin título): 0 (¡OK!)

 Resumen de Chunks por Documento:
+----------+------------------------------------------------------+------------+--------------+--------------+--------------+
|categoria |titulo_documento                                      |total_chunks|min_caracteres|avg_caracteres|max_caracteres|
+----------+------------------------------------------------------+------------+--------------+--------------+--------------+
|esg       |Esg Ib Informe Sostenibilidad                         |211         |75            |2778.0        |2799          |
|technical |Technical Guia Profesional Tramitacion Autoconsumo V.6|179         |1174          |2786.0        |2799          |
|legal     |Legal Reglamento 2016-679                             |94          |815           |2774.0        |2799          |
|operations|Operations Procedimiento Compras Prov